In [1]:
# Add this at the VERY TOP of Cell 2, before any imports
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [2]:
# 1: Set Up & Utilities

In [3]:
import os, time, random, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings("ignore")
 
# 🌱 Set seed globally
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass
 
# 🌱 Multi-seed wrapper
def run_with_seeds(train_fn, seeds=[42, 123, 7], **kwargs):
    metrics = {"acc": [], "f1": [], "train_time": [], "inf_time": []}
    raw_results = []
 
    for s in seeds:
        set_seed(s)
        res = train_fn(seed=s, **kwargs)
        raw_results.append({"seed": s, **res})
 
        for k in metrics:
            if f"test_{k}" in res:
                metrics[k].append(res[f"test_{k}"])
            elif k in res:
                metrics[k].append(res[k])
            else:
                raise KeyError(f"Missing metric '{k}' in result for seed {s}")
 
    summary = {
        k: {
            "mean": float(np.mean(v)),
            "std": float(np.std(v, ddof=1)) if len(v) > 1 else 0.0
        }
        for k, v in metrics.items()
    }
 
    return {"summary": summary, "raw_results": raw_results}
 
# 📏 Text Length & Truncation Analysis
def analyze_lengths(texts, tokenizer, max_len=512):
    lens = [len(tokenizer.encode(t, truncation=False)) for t in texts]
    stats = {
        "mean": float(np.mean(lens)),
        "median": float(np.median(lens)),
        "p95": float(np.percentile(lens, 95)),
        "max": int(np.max(lens)),
        "truncated": int(sum(l > max_len for l in lens)),
        "total": int(len(lens)),
        "truncated_pct": float(100 * sum(l > max_len for l in lens) / len(lens))
    }
 
    print(
        f"📊 Length Stats (tokens) | Mean: {stats['mean']:.1f} | "
        f"Median: {stats['median']:.1f} | P95: {stats['p95']:.1f} | Max: {stats['max']}"
    )
    print(
        f"🔪 {stats['truncated']}/{stats['total']} "
        f"({stats['truncated_pct']:.1f}%) exceed {max_len} tokens and will be truncated."
    )
    return stats
 
# ⚖️ Class Imbalance Check
def check_distribution(y, names):
    from collections import Counter
    cnt = Counter(y)
    total = len(y)
    rows = []
    for k, v in sorted(cnt.items()):
        rows.append({
            "class_id": k,
            "class_name": names[k],
            "count": v,
            "pct": 100 * v / total
        })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    return df

In [117]:
# 2: Load & Split Datasets

In [4]:
# 2A: Load & Split Newsgroups (via Kaggle Hub)
import os
import re
import kagglehub
from sklearn.model_selection import train_test_split

print("📥 Loading 20 Newsgroups from Kaggle...")

# Download dataset from Kaggle (no authentication required for public datasets)
base_path = kagglehub.dataset_download("crawford/20-newsgroups")

# Find all .txt files (each represents a newsgroup category)
txt_files = sorted([f for f in os.listdir(base_path) if f.endswith('.txt')])
if not txt_files:
    raise FileNotFoundError("No .txt files found in Kaggle download.")

# Create class names and label mapping
class_names = [f.replace('.txt', '') for f in txt_files]
label_map = {name: i for i, name in enumerate(class_names)}
print(f"📂 Found {len(class_names)} newsgroup files")

all_texts = []
all_labels = []

for cls in class_names:
    file_path = os.path.join(base_path, f"{cls}.txt")
    if not os.path.exists(file_path):
        continue
    
    with open(file_path, 'r', encoding='latin-1', errors='ignore') as f:
        raw = f.read()
    
    if not raw.strip():
        print(f"⚠️ {cls}.txt is empty. Skipping.")
        continue
    
    # Strategy 1: Split by double newline (handles \n\n and \r\n\r\n)
    docs = re.split(r'\r?\n\r?\n', raw)
    
    # Strategy 2: If Strategy 1 fails (<5 docs), split by Usenet header markers
    if len(docs) < 5:
        docs = re.split(r'(?=^From:|^Subject:|^Article-I\.D\.|^Path:)', raw, flags=re.MULTILINE)
    
    # Strategy 3: Fallback to single newline if still too few
    if len(docs) < 5:
        docs = raw.split('\n')
    
    docs_found = 0
    for doc in docs:
        doc = doc.strip()
        if len(doc) < 50:  # Skip very short documents
            continue
        
        # Light cleaning: remove header lines, keep body
        lines = doc.split('\n')
        cleaned = []
        in_header = True
        
        for line in lines:
            line_stripped = line.strip()
            
            if in_header:
                # Skip standard Usenet header fields
                if re.match(r'^(From|Subject|Organization|Date|Reply-To|Lines|Distribution|Keywords|Article-I\.D\.|NNTP-Posting-Host|Path|Message-ID|X-Newsreader|X-Trace|Posted|Followup-To):', line_stripped, re.I):
                    continue
                if line_stripped == '':
                    in_header = False
                    continue
                continue
            
            # Skip quotes/signatures
            if line_stripped.startswith('>') or line_stripped.startswith('|') or line_stripped.startswith('---') or line_stripped.startswith('___'):
                continue
            
            if line_stripped:
                cleaned.append(line_stripped)
        
        final_text = ' '.join(cleaned)
        if len(final_text) > 60:  # Keep only meaningful documents
            all_texts.append(final_text)
            all_labels.append(label_map[cls])
            docs_found += 1
    
    print(f"   📄 {cls}: {docs_found} docs loaded")

print(f"\n✅ Total 20NG documents: {len(all_texts)}")

if len(all_texts) == 0:
    # Fallback: print exact file stats so we can fix it in 1 message
    print("❌ Still zero documents. Printing file diagnostics:")
    print(f"   File: {file_path}")
    print(f"   Size: {len(raw)} bytes")
    print(f"   First 300 chars:\n{raw[:300]}")
    raise SystemExit("Reply with the diagnostics above so I can give you a 1-line fix.")

# Stratified Split: 80% Train, 10% Val, 10% Test
print("🔀 Splitting data (80/10/10)...")
X_train, X_temp, y_train, y_temp = train_test_split(
    all_texts, all_labels, 
    test_size=0.2, 
    random_state=42, 
    stratify=all_labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.5, 
    random_state=42, 
    stratify=y_temp
)

news_train = (X_train, y_train)
news_val = (X_val, y_val)
news_test = (X_test, y_test)
news_names = class_names

print(f"\n📊 20 Newsgroups Ready:")
print(f"   Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"   Sample text: '{X_train[0][:100]}...'")
print(f"   Sample label: {news_names[y_train[0]]}")

📥 Loading 20 Newsgroups from Kaggle...
📂 Found 20 newsgroup files
   📄 alt.atheism: 258 docs loaded
   📄 comp.graphics: 157 docs loaded
   📄 comp.os.ms-windows.misc: 140 docs loaded
   📄 comp.sys.ibm.pc.hardware: 120 docs loaded
   📄 comp.sys.mac.hardware: 142 docs loaded
   📄 comp.windows.x: 164 docs loaded
   📄 misc.forsale: 194 docs loaded
   📄 rec.autos: 90 docs loaded
   📄 rec.motorcycles: 118 docs loaded
   📄 rec.sport.baseball: 56 docs loaded
   📄 rec.sport.hockey: 204 docs loaded
   📄 sci.crypt: 228 docs loaded
   📄 sci.electronics: 114 docs loaded
   📄 sci.med: 206 docs loaded
   📄 sci.space: 128 docs loaded
   📄 soc.religion.christian: 138 docs loaded
   📄 talk.politics.guns: 170 docs loaded
   📄 talk.politics.mideast: 190 docs loaded
   📄 talk.politics.misc: 124 docs loaded
   📄 talk.religion.misc: 112 docs loaded

✅ Total 20NG documents: 3053
🔀 Splitting data (80/10/10)...

📊 20 Newsgroups Ready:
   Train: 2442 | Val: 305 | Test: 306
   Sample text: 'v)    To look into the 

In [119]:
# 2B: Load & Split TweeEval (Irony)

In [5]:
# 2B: Load & Split TweeEval (Irony)
import os
import glob
import kagglehub
import pandas as pd

print("📥 Loading TweetEval from Kaggle...")
path = kagglehub.dataset_download("thedevastator/tweeteval-a-multi-task-classification-benchmark")
print(f"📁 Dataset downloaded to: {path}")

# 1. Find all irony CSVs
all_csvs = glob.glob(os.path.join(path, "**/*.csv"), recursive=True)
irony_files = [f for f in all_csvs if "irony" in f.lower()]

if len(irony_files) < 3:
    raise ValueError(f"Could not find all 3 irony split files. Found: {[os.path.basename(f) for f in irony_files]}")

# 2. Robustly map split names to file paths
split_paths = {}
for f in irony_files:
    basename = os.path.basename(f).lower()
    if 'train' in basename and 'val' not in basename:
        split_paths['train'] = f
    elif 'validation' in basename or 'val' in basename:
        split_paths['validation'] = f
    elif 'test' in basename:
        split_paths['test'] = f

print(f"📄 Mapped splits: {list(split_paths.keys())}")

# 3. Load each split separately 
df_train = pd.read_csv(split_paths['train'])
df_val   = pd.read_csv(split_paths['validation'])
df_test  = pd.read_csv(split_paths['test'])

# 4. Identify columns dynamically
text_col = next((c for c in df_train.columns if c.lower() in ['text', 'tweet', 'sentence']), df_train.columns[0])
label_col = next((c for c in df_train.columns if c.lower() in ['label', 'class', 'target']), df_train.columns[1])
print(f"📊 Using column '{text_col}' for text and '{label_col}' for labels.")

tweet_names = ["non_ironic", "ironic"]

# 5. Clean & package into tuples (X, y)
def clean_split(df):
    df = df.dropna(subset=[text_col, label_col]).reset_index(drop=True)
    texts = df[text_col].astype(str).tolist()
    labels = df[label_col].astype(int).tolist()
    return (texts, labels)

tweet_train = clean_split(df_train)
tweet_val   = clean_split(df_val)
tweet_test  = clean_split(df_test)

print(f"\n✅ TweetEval Ready (Standard Splits):")
print(f"   Train: {len(tweet_train[0])} | Val: {len(tweet_val[0])} | Test: {len(tweet_test[0])}")
print(f"   Sample text: '{tweet_train[0][0][:80]}...'")
print(f"   Sample label: {tweet_names[tweet_train[1][0]]} (class {tweet_train[1][0]})")

📥 Loading TweetEval from Kaggle...
📁 Dataset downloaded to: C:\Users\JW\.cache\kagglehub\datasets\thedevastator\tweeteval-a-multi-task-classification-benchmark\versions\2
📄 Mapped splits: ['test', 'train', 'validation']
📊 Using column 'text' for text and 'label' for labels.

✅ TweetEval Ready (Standard Splits):
   Train: 2862 | Val: 955 | Test: 784
   Sample text: 'seeing ppl walking w/ crutches makes me really excited for the next 3 weeks of m...'
   Sample label: ironic (class 1)


 Tier 3 DistilBERT Training & Evaluation

In [7]:
#imports and device set up

import os, time, torch, numpy as np, shutil, pickle
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using NVIDIA GPU:", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon MPS")
else:
    device = torch.device("cpu")
    print("Using CPU (PyTorch CUDA not available in this environment)")



Using CPU (PyTorch CUDA not available in this environment)


In [8]:
#Setting up function

def train_transformer_tuned(model_name, X_train, y_train, X_val, y_val, X_test, y_test, num_labels, id2label, seed=42):
    set_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=False)

    max_len = 256 if num_labels > 2 else 128

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=max_len)

    train_ds = Dataset.from_dict({"text": X_train, "label": y_train}).map(
        tokenize, batched=True, remove_columns=["text"]
    )
    val_ds = Dataset.from_dict({"text": X_val, "label": y_val}).map(
        tokenize, batched=True, remove_columns=["text"]
    )
    test_ds = Dataset.from_dict({"text": X_test, "label": y_test}).map(
        tokenize, batched=True, remove_columns=["text"]
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            "acc": accuracy_score(labels, preds),
            "f1": f1_score(labels, preds, average="macro")
        }

    label2id = {v: k for k, v in id2label.items()}

    best_lr = 2e-5
    best_val_f1 = -1.0
    tune_dir = f"./tmp_tune_{model_name.replace('/', '_')}_{seed}"
    if os.path.exists(tune_dir):
        shutil.rmtree(tune_dir)

    print(f"Tuning {model_name} for seed {seed}...")
    for lr in [1e-5, 2e-5, 5e-5]:
        args = TrainingArguments(
            output_dir=tune_dir,
            learning_rate=lr,
            per_device_train_batch_size=4,
            per_device_eval_batch_size=8,
            num_train_epochs=2,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="eval_f1",
            save_total_limit=1,
            disable_tqdm=True,
            report_to="none",
            seed=seed,
            data_seed=seed,
            fp16=torch.cuda.is_available(),
            dataloader_num_workers=0
        )

        trainer = Trainer(
            model=AutoModelForSequenceClassification.from_pretrained(
                model_name,
                num_labels=num_labels,
                id2label=id2label,
                label2id=label2id
            ),
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
        )

        trainer.train()
        val_f1 = trainer.evaluate()["eval_f1"]

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_lr = lr

    print(f"Best LR for seed {seed}: {best_lr} | Val F1: {best_val_f1:.4f}")

    final_dir = f"./tmp_final_{model_name.replace('/', '_')}_{seed}"
    if os.path.exists(final_dir):
        shutil.rmtree(final_dir)

    args = TrainingArguments(
        output_dir=final_dir,
        learning_rate=best_lr,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1",
        save_total_limit=1,
        disable_tqdm=True,
        report_to="none",
        seed=seed,
        data_seed=seed,
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=0
    )

    trainer = Trainer(
        model=AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            id2label=id2label,
            label2id=label2id
        ),
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0

    t0 = time.time()
    test_res = trainer.evaluate(test_ds)
    inf_time = (time.time() - t0) / len(X_test)

    preds = np.argmax(trainer.predict(test_ds).predictions, axis=-1)
    report = classification_report(y_test, preds, output_dict=True, zero_division=0)
    cls_f1 = {id2label[i]: report[str(i)]["f1-score"] for i in range(num_labels) if str(i) in report}

    return {
        "seed": seed,
        "best_lr": best_lr,
        "test_acc": test_res["eval_acc"],
        "test_f1": test_res["eval_f1"],
        "train_time": train_time,
        "inf_time": inf_time,
        "preds": preds.tolist(),
        "best_class": max(cls_f1, key=cls_f1.get) if cls_f1 else "N/A",
        "worst_class": min(cls_f1, key=cls_f1.get) if cls_f1 else "N/A"
    }


Running Distilbert - each block with different seed

In [9]:
#Run 1 seed 42

RUN_SEED = 42
distil_seed42_results = {}

for ds_name, cfg in {
    "20NG": {
        "data": (news_train, news_val, news_test),
        "n": 20,
        "id2label": {i: str(i) for i in range(20)}
    },
    "TweetEval": {
        "data": (tweet_train, tweet_val, tweet_test),
        "n": 2,
        "id2label": {0: "non_ironic", 1: "ironic"}
    }
}.items():
    X_tr, y_tr = cfg["data"][0]
    X_val, y_val = cfg["data"][1]
    X_te, y_te = cfg["data"][2]

    print(f"\nDistilBERT on {ds_name} | Seed {RUN_SEED}")
    res = train_transformer_tuned(
        model_name="distilbert-base-uncased",
        X_train=X_tr, y_train=y_tr,
        X_val=X_val, y_val=y_val,
        X_test=X_te, y_test=y_te,
        num_labels=cfg["n"],
        id2label=cfg["id2label"],
        seed=RUN_SEED
    )

    key = f"{ds_name}_distilbert-base-uncased_seed42"
    distil_seed42_results[key] = res

with open("all_results_distil_seed42.pkl", "wb") as f:
    pickle.dump(distil_seed42_results, f)

print("Saved all_results_distil_seed42.pkl")



DistilBERT on 20NG | Seed 42


Map: 100%|██████████| 306/306 [00:00<00:00, 4326.27 examples/s]


Tuning distilbert-base-uncased for seed 42...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3435.50it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '2.562', 'grad_norm': '7.673', 'learning_rate': '5.917e-06', 'epoch': '0.8183'}
{'eval_loss': '1.999', 'eval_acc': '0.5377', 'eval_f1': '0.4243', 'eval_runtime': '29.79', 'eval_samples_per_second': '10.24', 'eval_steps_per_second': '1.309', 'epoch': '1'}


...

{'loss': '0.5486', 'grad_norm': '24.99', 'learning_rate': '1.07e-05', 'epoch': '1.397'}
{'eval_loss': '0.7694', 'eval_acc': '0.688', 'eval_f1': '0.6879', 'eval_runtime': '48.25', 'eval_samples_per_second': '19.8', 'eval_steps_per_second': '2.487', 'epoch': '2'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.83it/s]


{'loss': '0.4694', 'grad_norm': '17.07', 'learning_rate': '6.043e-06', 'epoch': '2.095'}
{'loss': '0.3464', 'grad_norm': '0.148', 'learning_rate': '1.387e-06', 'epoch': '2.793'}
{'eval_loss': '1.291', 'eval_acc': '0.6723', 'eval_f1': '0.6719', 'eval_runtime': '48.67', 'eval_samples_per_second': '19.62', 'eval_steps_per_second': '2.466', 'epoch': '3'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.17it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'train_runtime': '2046', 'train_samples_per_second': '4.197', 'train_steps_per_second': '1.05', 'train_loss': '0.4965', 'epoch': '3'}
{'eval_loss': '0.82', 'eval_acc': '0.6735', 'eval_f1': '0.6654', 'eval_runtime': '40.18', 'eval_samples_per_second': '19.51', 'eval_steps_per_second': '2.439', 'epoch': '3'}
Saved all_results_distil_seed42.pkl


In [10]:
#Run 2 : Seed 123

RUN_SEED = 123
distil_seed123_results = {}

for ds_name, cfg in {
    "20NG": {
        "data": (news_train, news_val, news_test),
        "n": 20,
        "id2label": {i: str(i) for i in range(20)}
    },
    "TweetEval": {
        "data": (tweet_train, tweet_val, tweet_test),
        "n": 2,
        "id2label": {0: "non_ironic", 1: "ironic"}
    }
}.items():
    X_tr, y_tr = cfg["data"][0]
    X_val, y_val = cfg["data"][1]
    X_te, y_te = cfg["data"][2]

    print(f"\nDistilBERT on {ds_name} | Seed {RUN_SEED}")
    res = train_transformer_tuned(
        model_name="distilbert-base-uncased",
        X_train=X_tr, y_train=y_tr,
        X_val=X_val, y_val=y_val,
        X_test=X_te, y_test=y_te,
        num_labels=cfg["n"],
        id2label=cfg["id2label"],
        seed=RUN_SEED
    )

    key = f"{ds_name}_distilbert-base-uncased_seed123"
    distil_seed123_results[key] = res

with open("all_results_distil_seed123.pkl", "wb") as f:
    pickle.dump(distil_seed123_results, f)

print("Saved all_results_distil_seed123.pkl")



DistilBERT on 20NG | Seed 123


Map: 100%|██████████| 306/306 [00:00<00:00, 3889.11 examples/s]


Tuning distilbert-base-uncased for seed 123...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5053.80it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '2.529', 'grad_norm': '11.25', 'learning_rate': '5.917e-06', 'epoch': '0.8183'}
{'eval_loss': '1.994', 'eval_acc': '0.5148', 'eval_f1': '0.3986', 'eval_runtime': '29.73', 'eval_samples_per_second': '10.26', 'eval_steps_per_second': '1.312', 'epoch': '1'}


...

{'loss': '0.5327', 'grad_norm': '12.94', 'learning_rate': '1.07e-05', 'epoch': '1.397'}
{'eval_loss': '0.8114', 'eval_acc': '0.6921', 'eval_f1': '0.6827', 'eval_runtime': '48.61', 'eval_samples_per_second': '19.65', 'eval_steps_per_second': '2.469', 'epoch': '2'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.41it/s]


{'loss': '0.4484', 'grad_norm': '0.4135', 'learning_rate': '6.043e-06', 'epoch': '2.095'}
{'loss': '0.284', 'grad_norm': '6.179', 'learning_rate': '1.387e-06', 'epoch': '2.793'}
{'eval_loss': '1.162', 'eval_acc': '0.689', 'eval_f1': '0.6889', 'eval_runtime': '50.66', 'eval_samples_per_second': '18.85', 'eval_steps_per_second': '2.369', 'epoch': '3'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.62it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'train_runtime': '2094', 'train_samples_per_second': '4.101', 'train_steps_per_second': '1.026', 'train_loss': '0.4724', 'epoch': '3'}
{'eval_loss': '1.208', 'eval_acc': '0.699', 'eval_f1': '0.6938', 'eval_runtime': '40.3', 'eval_samples_per_second': '19.45', 'eval_steps_per_second': '2.432', 'epoch': '3'}
Saved all_results_distil_seed123.pkl


In [ ]:
#run 3 : seed 7

RUN_SEED = 7
distil_seed7_results = {}

for ds_name, cfg in {
    "20NG": {
        "data": (news_train, news_val, news_test),
        "n": 20,
        "id2label": {i: str(i) for i in range(20)}
    },
    "TweetEval": {
        "data": (tweet_train, tweet_val, tweet_test),
        "n": 2,
        "id2label": {0: "non_ironic", 1: "ironic"}
    }
}.items():
    X_tr, y_tr = cfg["data"][0]
    X_val, y_val = cfg["data"][1]
    X_te, y_te = cfg["data"][2]

    print(f"\nDistilBERT on {ds_name} | Seed {RUN_SEED}")
    res = train_transformer_tuned(
        model_name="distilbert-base-uncased",
        X_train=X_tr, y_train=y_tr,
        X_val=X_val, y_val=y_val,
        X_test=X_te, y_test=y_te,
        num_labels=cfg["n"],
        id2label=cfg["id2label"],
        seed=RUN_SEED
    )

    key = f"{ds_name}_distilbert-base-uncased_seed7"
    distil_seed7_results[key] = res

with open("all_results_distil_seed7.pkl", "wb") as f:
    pickle.dump(distil_seed7_results, f)

print("Saved all_results_distil_seed7.pkl")
